In [156]:
import pandas as pd
#Api Banco Central de Chile
!pip install bcchapi
import bcchapi
import requests
import zipfile
import tempfile
import os
from io import BytesIO
import time
from tqdm import tqdm
import itertools

### Sociedad Hipotecaria Federal (Mexico)

In [ ]:
shf=pd.read_excel("https://www.gob.mx/cms/uploads/attachment/file/1055836/Indice_SHF_datos_abiertos_4_trim_2025.xlsx")
shf.columns=shf.columns.str.lower()

shf=shf[["global", "estado","municipio", "trimestre", "año","indice"]]
#Rename
shf=shf.rename(columns={"global":"variable", "estado":"state", "municipio":"municipality", "trimestre":"quarter", "año":"year", "indice":"value"})
shf['date'] = pd.PeriodIndex(
    year=shf['year'],
    quarter=shf['quarter'],
    freq='Q'
).to_timestamp()
#Add country
shf["country"]="Mexico"
#add source
shf["source"]="Sociedad Hipotecaria Federal"
shf["base_year"]=2017
shf['var_type'] = 'index'
shf=shf[["country", "state", "municipality", "date", "variable", "value","var_type", "source", "base_year"]]

shf

C:\Users\claud\AppData\Local\Temp\ipykernel_51420\904601658.py:7: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  shf['date'] = pd.PeriodIndex(


,country,state,municipality,date,variable,value,var_type,source,base_year
0,Mexico,NaN,NaN,2005-01-01,Nacional,48.47,index,Sociedad Hipotecaria Federal,2017
1,Mexico,NaN,NaN,2005-01-01,Nueva,48.20,index,Sociedad Hipotecaria Federal,2017
2,Mexico,NaN,NaN,2005-01-01,Usada,48.76,index,Sociedad Hipotecaria Federal,2017
3,Mexico,NaN,NaN,2005-01-01,Casa sola,49.08,index,Sociedad Hipotecaria Federal,2017
4,Mexico,NaN,NaN,2005-01-01,Casa en condominio - depto.,47.79,index,Sociedad Hipotecaria Federal,2017
...,...,...,...,...,...,...,...,...,...
10159,Mexico,Veracruz,Veracruz,2025-10-01,NaN,194.41,index,Sociedad Hipotecaria Federal,2017
10160,Mexico,Yucatán,Kanasín,2025-10-01,NaN,206.26,index,Sociedad Hipotecaria Federal,2017
10161,Mexico,Yucatán,Mérida,2025-10-01,NaN,209.36,index,Sociedad Hipotecaria Federal,2017
10162,Mexico,Zacatecas,Guadalupe,2025-10-01,NaN,180.85,index,Sociedad Hipotecaria Federal,2017


### Banco Central de Chile (Chile)

In [57]:
creds = {}

with open("C:\\Users\\claud\\Documents\\credenciales_bcc.txt", "r") as f:
    for linea in f:
        clave, valor = linea.strip().split("=")
        creds[clave] = valor

usuario = creds["usuario"]
password = creds["password"]

credenciales = bcchapi.Siete(usuario, password)


In [84]:
tabla_ipv=credenciales.cuadro(
  series=["F034.IPV.FLU.BCCH.2008.0.T", 
          "F034.IPVC.FLU.BCCH.2008.0.T",
          "F034.IPVCNEW.FLU.BCCH.2008.0.T",
          "F034.IPVCUSED.FLU.BCCH.2008.0.T",
          "F034.IPVD.FLU.BCCH.2008.0.T", 
          "F034.IPVDNEW.FLU.BCCH.2008.0.T",
          "F034.IPVDUSED.FLU.BCCH.2008.0.T",
          "F034.IVPZ1.FLU.BCCH.2008.0.T",
          "F034.IPVZ2.FLU.BCCH.2008.0.T",
          "F034.IPVZ3.FLU.BCCH.2008.0.T",
          "F034.IPVZ4.FLU.BCCH.2008.0.T"
        
          ],
  nombres = ["ipv_general", 
             "ipv_houses", 
             "ipv_new_houses",
             "ipv_used_houses",
              "ipv_apartments",
              "ipv_new_apartments",
              "ipv_used_apartments",
              "ipv_north_zone",
              "ipv_central_zone",
              "ipv_south_zone",
              "ipv_metropolitan_region"
             
             
             
             
             
             ],
  desde="2001-01-01",
  hasta="2025-12-31"
)
tabla_ipv = tabla_ipv.reset_index().rename(columns={'index': 'date'})
tabla_ipv = tabla_ipv.melt(
    id_vars='date',          
    var_name='variable',     
    value_name='value'       
)
tabla_ipv["country"]="Chile"
tabla_ipv["source"]="Banco Central de Chile"
tabla_ipv["base_year"]=2008
tabla_ipv['var_type'] = 'index'
tabla_ipv

,date,variable,value,country,source,base_year,var_type
0,2002-01-01,ipv_general,79.819860,Chile,Banco Central de Chile,2008,index
1,2002-04-01,ipv_general,82.868177,Chile,Banco Central de Chile,2008,index
2,2002-07-01,ipv_general,83.698377,Chile,Banco Central de Chile,2008,index
3,2002-10-01,ipv_general,83.196206,Chile,Banco Central de Chile,2008,index
4,2003-01-01,ipv_general,82.737826,Chile,Banco Central de Chile,2008,index
...,...,...,...,...,...,...,...
1040,2024-07-01,ipv_metropolitan_region,228.655103,Chile,Banco Central de Chile,2008,index
1041,2024-10-01,ipv_metropolitan_region,230.045391,Chile,Banco Central de Chile,2008,index
1042,2025-01-01,ipv_metropolitan_region,232.929202,Chile,Banco Central de Chile,2008,index
1043,2025-04-01,ipv_metropolitan_region,234.278572,Chile,Banco Central de Chile,2008,index


#### Banco de la República (Colombia)

In [85]:
url="https://suameca.banrep.gov.co/estadisticas-economicas-back/rest/estadisticaEconomicaRestService/exportarReporteExcel?reportPath=/shared/Estadisticas_Banco_de_la_Republica/1_Precios_e_Inflacion/5_Indice_de_precios_de_la_vivienda/2_IPVU_Indice_de_precios_de_la_vivienda_usada/2_IPVU_Indice_nominal_y_real_periodicidad_trimestral_iqy&formato=csv"

In [86]:
ipvu=pd.read_csv(url)
#delete first column
ipvu=ipvu.drop(ipvu.columns[0], axis=1)
#rename columns
ipvu=ipvu.rename(columns={"Fecha":"date", "índice nominal":"ipvu_nominal", "Índice real":"ipvu_real"})
ipvu = ipvu.melt(
    id_vars='date',          
    var_name='variable',     
    value_name='index'       
)
ipvu["country"]="Colombia"
ipvu["source"]="Banco de la República Colombia"
ipvu["base_year"]=1990
ipvu['var_type'] = 'index'
ipvu

,date,variable,index,country,source,base_year,var_type
0,1988-03-31 00:00:00,Índice nominal,55.602458,Colombia,Banco de la República Colombia,1990,index
1,1988-06-30 00:00:00,Índice nominal,54.278016,Colombia,Banco de la República Colombia,1990,index
2,1988-09-30 00:00:00,Índice nominal,64.241675,Colombia,Banco de la República Colombia,1990,index
3,1988-12-31 00:00:00,Índice nominal,63.775098,Colombia,Banco de la República Colombia,1990,index
4,1989-03-31 00:00:00,Índice nominal,72.866472,Colombia,Banco de la República Colombia,1990,index
...,...,...,...,...,...,...,...
297,2024-09-30 00:00:00,ipvu_real,125.463418,Colombia,Banco de la República Colombia,1990,index
298,2024-12-31 00:00:00,ipvu_real,123.697949,Colombia,Banco de la República Colombia,1990,index
299,2025-03-31 00:00:00,ipvu_real,124.081023,Colombia,Banco de la República Colombia,1990,index
300,2025-06-30 00:00:00,ipvu_real,128.251738,Colombia,Banco de la República Colombia,1990,index


In [121]:

cookies = {
    'ASP.NET_SessionId': 'nlozsogjk5ww4cqxqg1jpxnv',
    '__RequestVerificationToken': '7BCLDlTekEDtACKollit9w2ymsNm-OL0yPRKX3spSLP-pJTX4PdkAsQ7aPJcGZCTl6TAKfCh62Wt8ys23vrd99uGhTLvyFo4PkdeCHft9qU1',
    'ROUTEID': '.5',
    '_gcl_au': '1.1.528231028.1775600956',
    '_gid': 'GA1.3.259430437.1775789301',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    '_ga': 'GA1.1.1370069149.1775600956',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjJiNTA0MDhmZS0zZDdlLTRmYmItYjNiNi00OGQzMWQxZTEwYmElMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc1NzkzNTg5OTk3JTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlMkMlMjJsYXN0RXZlbnRUaW1lJTIyJTNBMTc3NTc5MzU5MDAwNiU3RA==',
    '_ga_WN7SEBCDBR': 'GS2.1.s1775793590$o5$g1$t1775794876$j60$l0$h0',
}

headers = {
    'accept': '*/*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'origin': 'https://www.fipe.org.br',
    'priority': 'u=1, i',
    'referer': 'https://www.fipe.org.br/en-us/indexes-and-indicators/fipezap/',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-requested-with': 'XMLHttpRequest',
    # 'cookie': 'ASP.NET_SessionId=nlozsogjk5ww4cqxqg1jpxnv; __RequestVerificationToken=7BCLDlTekEDtACKollit9w2ymsNm-OL0yPRKX3spSLP-pJTX4PdkAsQ7aPJcGZCTl6TAKfCh62Wt8ys23vrd99uGhTLvyFo4PkdeCHft9qU1; ROUTEID=.5; _gcl_au=1.1.528231028.1775600956; _gid=GA1.3.259430437.1775789301; AMP_MKTG_8f1ede8e9c=JTdCJTdE; _ga=GA1.1.1370069149.1775600956; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjJiNTA0MDhmZS0zZDdlLTRmYmItYjNiNi00OGQzMWQxZTEwYmElMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc1NzkzNTg5OTk3JTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlMkMlMjJsYXN0RXZlbnRUaW1lJTIyJTNBMTc3NTc5MzU5MDAwNiU3RA==; _ga_WN7SEBCDBR=GS2.1.s1775793590$o5$g1$t1775794876$j60$l0$h0',
}

data = {
    'codigoTipo': '2',
    'codigoInfo': '2',
}

response1 = requests.post('https://www.fipe.org.br/IndicesConsulta-FipeZapRegiao', cookies=cookies, headers=headers, data=data)

In [122]:
regions1=pd.DataFrame(response1.json())
regions1.columns=regions1.columns.str.lower()
#sort by codigo
regions1=regions1.sort_values(by="codigo")
regions1

,codigo,nome
52,2,São Paulo
39,3,Rio de Janeiro
4,4,Belo Horizonte
15,5,Distrito Federal
37,6,Recife
18,7,Fortaleza
40,8,Salvador
42,9,Santo André
44,10,São Bernardo do Campo
45,11,São Caetano do Sul


In [123]:
cookies = {
    'ASP.NET_SessionId': 'nlozsogjk5ww4cqxqg1jpxnv',
    '__RequestVerificationToken': '7BCLDlTekEDtACKollit9w2ymsNm-OL0yPRKX3spSLP-pJTX4PdkAsQ7aPJcGZCTl6TAKfCh62Wt8ys23vrd99uGhTLvyFo4PkdeCHft9qU1',
    'ROUTEID': '.5',
    '_gcl_au': '1.1.528231028.1775600956',
    '_gid': 'GA1.3.259430437.1775789301',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    '_ga': 'GA1.1.1370069149.1775600956',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjJiNTA0MDhmZS0zZDdlLTRmYmItYjNiNi00OGQzMWQxZTEwYmElMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc1NzkzNTg5OTk3JTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlMkMlMjJsYXN0RXZlbnRUaW1lJTIyJTNBMTc3NTc5MzU5MDAwNiU3RA==',
    '_ga_WN7SEBCDBR': 'GS2.1.s1775793590$o5$g1$t1775795823$j60$l0$h0',
}

headers = {
    'accept': '*/*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'origin': 'https://www.fipe.org.br',
    'priority': 'u=1, i',
    'referer': 'https://www.fipe.org.br/en-us/indexes-and-indicators/fipezap/',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-requested-with': 'XMLHttpRequest',
    # 'cookie': 'ASP.NET_SessionId=nlozsogjk5ww4cqxqg1jpxnv; __RequestVerificationToken=7BCLDlTekEDtACKollit9w2ymsNm-OL0yPRKX3spSLP-pJTX4PdkAsQ7aPJcGZCTl6TAKfCh62Wt8ys23vrd99uGhTLvyFo4PkdeCHft9qU1; ROUTEID=.5; _gcl_au=1.1.528231028.1775600956; _gid=GA1.3.259430437.1775789301; AMP_MKTG_8f1ede8e9c=JTdCJTdE; _ga=GA1.1.1370069149.1775600956; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjJiNTA0MDhmZS0zZDdlLTRmYmItYjNiNi00OGQzMWQxZTEwYmElMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc1NzkzNTg5OTk3JTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlMkMlMjJsYXN0RXZlbnRUaW1lJTIyJTNBMTc3NTc5MzU5MDAwNiU3RA==; _ga_WN7SEBCDBR=GS2.1.s1775793590$o5$g1$t1775795823$j60$l0$h0',
}

data = {
    'codigoTipo': '1',
    'codigoInfo': '2',
}

response2 = requests.post('https://www.fipe.org.br/IndicesConsulta-FipeZapRegiao', cookies=cookies, headers=headers, data=data)

In [124]:
regions2=pd.DataFrame(response2.json())
regions2.columns=regions2.columns.str.lower()
#sort by codigo
regions2=regions2.sort_values(by="codigo")
regions2

,codigo,nome
34,2,São Paulo
25,3,Rio de Janeiro
3,4,Belo Horizonte
8,5,Distrito Federal
23,6,Recife
11,7,Fortaleza
26,8,Salvador
27,9,Santo André
29,10,São Bernardo do Campo
19,12,Niterói


In [142]:
#Concat regions1 and regions2 and drop duplicates
regions=pd.concat([regions1, regions2], ignore_index=True).drop_duplicates()
#sort by codigo
regions=regions.sort_values(by="codigo")
regions

,codigo,nome
0,2,São Paulo
1,3,Rio de Janeiro
2,4,Belo Horizonte
3,5,Distrito Federal
4,6,Recife
5,7,Fortaleza
6,8,Salvador
7,9,Santo André
8,10,São Bernardo do Campo
9,11,São Caetano do Sul


In [ ]:
def get_fipezap(tipo, region, dormitorios):

    url = (
        "https://www.fipe.org.br/IndicesConsulta-FipeZapPesquisa"
        f"?tipo={tipo}&regiao={region}&info=2&dormitorio={dormitorios}"
    )

    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, headers=headers, timeout=10)
    r.raise_for_status()

    data = r.json()
    df = pd.DataFrame(data)

    if df.empty:
        return None

    #Drop last column
    df = df.loc[:, ~df.columns.str.contains("Variacao", case=False)]

    value_col = [c for c in df.columns if c not in [
        'Ano', 'Mes', 'CodigoDormitorio'
    ]][0]

    # rename
    df = df.rename(columns={
        "Ano": "year",
        "Mes": "month",
        "CodigoDormitorio": "rooms",
        value_col: "value"
    })

    # date
    df["date"] = pd.to_datetime(dict(year=df.year, month=df.month, day=1))

    # dorms
    room_map = {
        1: 'total',
        2: '1',
        3: '2',
        4: '3',
        5: '4+'
    }

    df['rooms'] = df['rooms'].map(room_map)

    # metadata
    df['region_code'] = region
    df['variable'] = 'price_sale_index' if tipo == 2 else 'price_rent_index'
    df['var_type'] = 'index'
    df['country'] = 'Brazil'
    df['source'] = 'FIPEZAP'

    return df[['date','region_code','rooms','variable','value','var_type','country','source']]

#Loop

def load_fipezap_all(df_regiones):

    tipos = [1, 2]        
    dorms = [1,2,3,4,5]

    dfs = []
    errors = 0

    # combinaciones completas
    combos = list(itertools.product(df_regiones['codigo'], tipos, dorms))

    for region, tipo, dorm in tqdm(combos, desc="Downloading FIPEZAP"):

        try:
            df_tmp = get_fipezap(tipo, region, dorm)

            if df_tmp is not None:
                dfs.append(df_tmp)

            time.sleep(0.2)  

        except Exception as e:
            errors += 1
            print(f"\nERROR region={region} tipo={tipo} dorm={dorm} -> {e}")

    # concat
    df_final = pd.concat(dfs, ignore_index=True)

    # merge with regions
    df_final = df_final.merge(
        df_regiones,
        left_on='region_code',
        right_on='codigo',
        how='left'
    )

    # city
    df_final = df_final.rename(columns={'nome': 'city'})

    # 
    df_final = df_final[[
        'date','country','region_code','city',
        'rooms','variable','value',
        'var_type','source'
    ]]

    print(f"\nErrores totales: {errors}")

    return df_final

In [159]:
# Get data
df_fipezap = load_fipezap_all(regions)

df_fipezap.head()


ERROR region=7 tipo=1 dorm=2 -> If using all scalar values, you must pass an index



ERROR region=7 tipo=1 dorm=3 -> If using all scalar values, you must pass an index



ERROR region=7 tipo=1 dorm=4 -> If using all scalar values, you must pass an index



ERROR region=7 tipo=1 dorm=5 -> If using all scalar values, you must pass an index



ERROR region=7 tipo=2 dorm=2 -> If using all scalar values, you must pass an index


...


ERROR region=59 tipo=2 dorm=2 -> If using all scalar values, you must pass an index



ERROR region=59 tipo=2 dorm=3 -> If using all scalar values, you must pass an index



ERROR region=59 tipo=2 dorm=4 -> If using all scalar values, you must pass an index



ERROR region=59 tipo=2 dorm=5 -> If using all scalar values, you must pass an index

Errores totales: 333


,date,country,region_code,city,rooms,variable,value,var_type,source
0,2026-02-01,Brazil,2,São Paulo,total,price_rent_index,241.25063,index,FIPEZAP
1,2026-01-01,Brazil,2,São Paulo,total,price_rent_index,240.02878,index,FIPEZAP
2,2025-12-01,Brazil,2,São Paulo,total,price_rent_index,239.00537,index,FIPEZAP
3,2025-11-01,Brazil,2,São Paulo,total,price_rent_index,237.83917,index,FIPEZAP
4,2025-10-01,Brazil,2,São Paulo,total,price_rent_index,236.52854,index,FIPEZAP


In [161]:
df_fipezap["base_year"]=2010
df_fipezap

,date,country,region_code,city,rooms,variable,value,var_type,source,base_year
0,2026-02-01,Brazil,2,São Paulo,total,price_rent_index,241.25063,index,FIPEZAP,2010
1,2026-01-01,Brazil,2,São Paulo,total,price_rent_index,240.02878,index,FIPEZAP,2010
2,2025-12-01,Brazil,2,São Paulo,total,price_rent_index,239.00537,index,FIPEZAP,2010
3,2025-11-01,Brazil,2,São Paulo,total,price_rent_index,237.83917,index,FIPEZAP,2010
4,2025-10-01,Brazil,2,São Paulo,total,price_rent_index,236.52854,index,FIPEZAP,2010
...,...,...,...,...,...,...,...,...,...,...
28620,2022-05-01,Brazil,59,Belém,total,price_sale_index,98.31647,index,FIPEZAP,2010
28621,2022-04-01,Brazil,59,Belém,total,price_sale_index,99.09569,index,FIPEZAP,2010
28622,2022-03-01,Brazil,59,Belém,total,price_sale_index,100.08294,index,FIPEZAP,2010
28623,2022-02-01,Brazil,59,Belém,total,price_sale_index,100.66682,index,FIPEZAP,2010


In [137]:
url="https://www.fipe.org.br/IndicesConsulta-FipeZapPesquisa?tipo=1&regiao=28&info=2&dormitorio=5"

response = requests.get(url, headers=headers)
print(response.status_code)
data=pd.DataFrame(response.json())
data

200


,Ano,Mes,CodigoDormitorio,FipeZap Locação,FipeZap Locação_VariacaoUltimos12Meses
0,2026,2,5,259.267830,8.967574
1,2026,1,5,257.402410,9.897506
2,2025,12,5,254.112660,9.640962
3,2025,11,5,250.545210,9.229440
4,2025,10,5,248.215600,9.370148
...,...,...,...,...,...
129,2015,5,5,146.837009,-2.922848
130,2015,4,5,147.590304,-3.289859
131,2015,3,5,146.977171,-3.665003
132,2015,2,5,147.322498,-2.592887


In [168]:
#Paste all dataframes together
indices_vivienda=pd.concat([tabla_ipv, ipvu,shf,df_fipezap], ignore_index=True)
#Save to csv
indices_vivienda.to_csv("housing_indexes.csv",encoding="latin1", index=False)
indices_vivienda

,date,variable,value,country,source,base_year,var_type,index,state,municipality,region_code,city,rooms
0,2002-01-01 00:00:00,ipv_general,79.819860,Chile,Banco Central de Chile,2008,index,NaN,NaN,NaN,NaN,NaN,NaN
1,2002-04-01 00:00:00,ipv_general,82.868177,Chile,Banco Central de Chile,2008,index,NaN,NaN,NaN,NaN,NaN,NaN
2,2002-07-01 00:00:00,ipv_general,83.698377,Chile,Banco Central de Chile,2008,index,NaN,NaN,NaN,NaN,NaN,NaN
3,2002-10-01 00:00:00,ipv_general,83.196206,Chile,Banco Central de Chile,2008,index,NaN,NaN,NaN,NaN,NaN,NaN
4,2003-01-01 00:00:00,ipv_general,82.737826,Chile,Banco Central de Chile,2008,index,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40131,2022-05-01 00:00:00,price_sale_index,98.316470,Brazil,FIPEZAP,2010,index,NaN,NaN,NaN,59.0,Belém,total
40132,2022-04-01 00:00:00,price_sale_index,99.095690,Brazil,FIPEZAP,2010,index,NaN,NaN,NaN,59.0,Belém,total
40133,2022-03-01 00:00:00,price_sale_index,100.082940,Brazil,FIPEZAP,2010,index,NaN,NaN,NaN,59.0,Belém,total
40134,2022-02-01 00:00:00,price_sale_index,100.666820,Brazil,FIPEZAP,2010,index,NaN,NaN,NaN,59.0,Belém,total


### Río de Janeiro (Brazil)

In [171]:
base_url = "https://pgeo3.rio.rj.gov.br/arcgis/rest/services/Fazenda/ITBI/MapServer/3/query"

params = {
    "where": "1=1",
    "outFields": "*",
    "outSR": 4326,
    "f": "json",
    "resultRecordCount": 1000  
}

todos = []
offset = 0

while True:
    params["resultOffset"] = offset
    
    r = requests.get(base_url, params=params)
    data = r.json()
    
    features = data.get("features", [])
    
    if not features:
        break
    
    todos.extend([f["attributes"] for f in features])
    

    if not data.get("exceededTransferLimit", False):
        break
    
    offset += 1000

rio = pd.DataFrame(todos)
rio["source"] = "Prefeitura da Cidade do Rio de Janeiro"
rio.to_csv("transactions_riodejaneiro.csv",encoding="latin1",index=False)
rio

,objectid,cl,logradouro,codbairro,bairro,total_transações,uso,média_percentual_transferido,média_área_terreno,média_valor_transação,média_valor_imóvel,principal_transação_mercado,ano_transação,source
0,1328978,126680,RUA CHOVE LA FORA,151,Guaratiba,2,TERRITORIAL,100.0,300.0,26136.010000,26136.010000,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
1,1328979,135830,RUA ENRICO CARUSO,151,Guaratiba,2,TERRITORIAL,100.0,360.0,64745.990000,64745.990000,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
2,1328980,136028,RUA TORRE DE PEDRA,151,Guaratiba,2,TERRITORIAL,75.0,345.0,51107.765000,65957.770000,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
3,1328981,136036,RUA VARGEM DO CEDRO,151,Guaratiba,2,TERRITORIAL,100.0,375.0,77932.790000,77932.790000,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
4,1328982,225391,RUA FRANCISCO MIGNONE,151,Guaratiba,3,TERRITORIAL,100.0,416.0,167867.376666,167867.376666,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4735,1333713,033431,ETR DA PACIENCIA,148,Paciência,2,TERRITORIAL,100.0,297.0,46935.885000,46935.885000,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
4736,1333714,001008,RUA D-2 PAL 19600,151,Guaratiba,3,TERRITORIAL,100.0,360.0,51811.213333,51811.213333,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
4737,1333715,031286,ETR DA MATRIZ,151,Guaratiba,5,TERRITORIAL,100.0,5012.0,286526.596000,286526.596000,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
4738,1333716,125773,RUA LUIZ BRUNINI (RADIALISTA),151,Guaratiba,3,TERRITORIAL,100.0,300.0,27181.440000,27181.440000,COMPRA E VENDA,2021,Prefeitura da Cidade do Rio de Janeiro
